In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import random
import string
import re
import os
import shutil

# Initialize Spark
spark = SparkSession.builder \
    .appName("TPCH_DataQualityIssues") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

class TPCHErrorInjector:
    """Inject controllable data quality issues into TPC-H tables"""
    
    def __init__(self, spark_session, error_rate=0.1):
        self.spark = spark_session
        self.error_rate = error_rate  # Probability of introducing an error
        self.phone_prefixes = ["+1", "+44", "+91", "+86", "+81", "+49", "+33", "+61"]
        
    def add_leading_trailing_spaces(self, df, string_columns):
        """Add leading/trailing spaces to string columns"""
        for col_name in string_columns:
            if col_name in df.columns:
                df = df.withColumn(
                    col_name,
                    when(rand() < self.error_rate,
                         concat(lit(" "), col(col_name), lit("  ")))
                    .otherwise(col(col_name))
                )
        return df
    
    def add_special_characters(self, df, string_columns):
        """Add special characters to text fields"""
        special_chars = ["@", "#", "$", "%", "&", "*", "!", "?"]
        
        for col_name in string_columns:
            if col_name in df.columns:
                # Randomly select a special character using rand() and case when
                df = df.withColumn(
                    col_name,
                    when(rand() < self.error_rate,
                         concat(col(col_name), 
                                when(rand() < 0.125, lit("@"))
                                .when(rand() < 0.25, lit("#"))
                                .when(rand() < 0.375, lit("$"))
                                .when(rand() < 0.5, lit("%"))
                                .when(rand() < 0.625, lit("&"))
                                .when(rand() < 0.75, lit("*"))
                                .when(rand() < 0.875, lit("!"))
                                .otherwise(lit("?"))))
                    .otherwise(col(col_name))
                )
        return df
    
    def mix_case_inconsistency(self, df, string_columns):
        """Randomly change case of string columns"""
        for col_name in string_columns:
            if col_name in df.columns:
                df = df.withColumn(
                    col_name,
                    when(rand() < self.error_rate,
                         when(rand() < 0.5, upper(col(col_name)))
                         .otherwise(lower(col(col_name))))
                    .otherwise(col(col_name))
                )
        return df
    
    def add_null_values(self, df, columns, null_rate=0.03):
        """Introduce NULL values in specified columns"""
        for col_name in columns:
            if col_name in df.columns:
                df = df.withColumn(
                    col_name,
                    when(rand() < null_rate, lit(None))
                    .otherwise(col(col_name))
                )
        return df
    
    def add_duplicate_records(self, df, duplicate_rate=0.03):
        """Add duplicate records to the dataset"""
        duplicates = df.sample(fraction=duplicate_rate, seed=42)
        return df.union(duplicates)
    
    def corrupt_numeric_values(self, df, numeric_columns, corrupt_rate=0.02):
        """Corrupt numeric values by adding random noise or setting to extremes"""
        for col_name in numeric_columns:
            if col_name in df.columns:
                df = df.withColumn(
                    col_name,
                    when(rand() < corrupt_rate,
                         when(rand() < 0.33, col(col_name) * (rand() * 10 + 0.5))  # Scale
                         .when(rand() < 0.66, col(col_name) * -1)  # Negative
                         .otherwise(lit(None)))  # NULL
                    .otherwise(col(col_name))
                )
        return df

def write_single_parquet(df, output_path):
    """Write DataFrame as a single parquet file (not a directory)"""
    # Create temporary directory
    temp_dir = output_path + "_temp"
    
    # Write as single parquet file (Spark always creates a directory)
    df.coalesce(1).write.mode("overwrite").parquet(temp_dir)
    
    # Find the actual parquet file in the temp directory
    for file in os.listdir(temp_dir):
        if file.endswith('.parquet'):
            # Move the parquet file to the desired location
            shutil.move(os.path.join(temp_dir, file), output_path)
            break
    
    # Remove the temporary directory
    shutil.rmtree(temp_dir)
    
    return output_path

def apply_selected_error_types(df, table_context):
    """Apply ONLY the selected error types based on table context"""
    
    # Define table-specific configurations based on EDA findings
    if table_context == "orders":
        # Add leading/trailing spaces (present in o_comment)
        df = error_injector.add_leading_trailing_spaces(df, ["o_comment"])
        
        # Add special characters (present in o_comment)
        df = error_injector.add_special_characters(df, ["o_comment"])
        
        # Mix case inconsistency
        df = error_injector.mix_case_inconsistency(df, ["o_clerk"])
        
        # Add NULL values
        df = error_injector.add_null_values(df, ["o_comment"], null_rate=0.03)
        
        # Add duplicate records
        df = error_injector.add_duplicate_records(df, duplicate_rate=0.03)
        
        # Corrupt numeric values (o_totalprice)
        df = error_injector.corrupt_numeric_values(df, ["o_totalprice"], corrupt_rate=0.02)
        
    elif table_context == "lineitem":
        # Add leading/trailing spaces (present in l_comment)
        df = error_injector.add_leading_trailing_spaces(df, ["l_comment"])
        
        # Add special characters (present in l_comment)
        df = error_injector.add_special_characters(df, ["l_comment"])
        
        # Add NULL values
        df = error_injector.add_null_values(df, ["l_comment"], null_rate=0.03)
        
        # Add duplicate records
        df = error_injector.add_duplicate_records(df, duplicate_rate=0.03)
        
        # Corrupt numeric values (l_quantity, l_discount, l_tax)
        df = error_injector.corrupt_numeric_values(df, ["l_quantity", "l_discount", "l_tax"], corrupt_rate=0.02)
    
    return df

# Create error injector instance
error_injector = TPCHErrorInjector(spark, error_rate=0.15)

def corrupt_incremental_data():
    """Apply corruption to incremental_ingestion_combined data and save to corrupted_incremental"""
    
    # Define paths
    source_path = "../../../data/raw/tables/incremental_ingestion_combined"
    corrupted_path = "../../../data/raw/tables/corrupted_incremental"
    
    if not os.path.exists(source_path):
        print(f"❌ Error: Source path {source_path} does not exist!")
        return
    
    print("\n" + "="*80)
    print("🔧 CORRUPTING INCREMENTAL INGESTION COMBINED DATA")
    print("="*80)
    print(f"Source: {source_path}")
    print(f"Destination: {corrupted_path}")
    
    # Create corrupted directory structure
    if os.path.exists(corrupted_path):
        print(f"\n⚠️ Warning: {corrupted_path} already exists.")
        response = input("Do you want to overwrite it? (yes/no): ")
        if response.lower() != 'yes':
            print("❌ Operation cancelled.")
            return
        shutil.rmtree(corrupted_path)
    
    os.makedirs(corrupted_path)
    
    # Track corruption statistics
    corruption_stats = []
    
    # 1. Process Base Layer
    print("\n" + "="*80)
    print("📊 PROCESSING BASE LAYER")
    print("="*80)
    
    bases_path = f"{source_path}/bases"
    corrupted_bases_path = f"{corrupted_path}/bases"
    os.makedirs(corrupted_bases_path, exist_ok=True)
    
    if os.path.exists(bases_path):
        for file_name in os.listdir(bases_path):
            if file_name.endswith('.parquet'):
                file_path = f"{bases_path}/{file_name}"
                
                # Determine table type from filename
                if 'orders' in file_name:
                    table_type = 'orders'
                elif 'lineitem' in file_name:
                    table_type = 'lineitem'
                else:
                    # Skip other base files
                    continue
                
                print(f"\n📁 Processing base file: {file_name}")
                print(f"   Table type: {table_type}")
                
                try:
                    # Load original data
                    df_original = spark.read.parquet(file_path)
                    original_count = df_original.count()
                    print(f"   Original records: {original_count:,}")
                    
                    # Apply corruption
                    df_corrupted = apply_selected_error_types(df_original, table_type)
                    corrupted_count = df_corrupted.count()
                    
                    # Save corrupted data as single parquet file
                    output_path = f"{corrupted_bases_path}/{file_name}"
                    write_single_parquet(df_corrupted, output_path)
                    
                    print(f"   ✅ Corrupted data saved to: {output_path}")
                    print(f"   Records: {original_count:,} → {corrupted_count:,} (+{corrupted_count - original_count:,})")
                    
                    corruption_stats.append({
                        "layer": "base",
                        "file": file_name,
                        "original": original_count,
                        "corrupted": corrupted_count,
                        "increase": corrupted_count - original_count
                    })
                    
                except Exception as e:
                    print(f"   ❌ Error processing {file_name}: {e}")
    
    # 2. Process Incremental Batches
    print("\n" + "="*80)
    print("📊 PROCESSING INCREMENTAL BATCHES")
    print("="*80)
    
    increments_path = f"{source_path}/increments"
    corrupted_increments_path = f"{corrupted_path}/increments"
    os.makedirs(corrupted_increments_path, exist_ok=True)
    
    if os.path.exists(increments_path):
        # Iterate through batches
        for batch_num in range(1, 4):
            batch_name = f"batch_{batch_num}"
            batch_source = f"{increments_path}/{batch_name}"
            batch_dest = f"{corrupted_increments_path}/{batch_name}"
            
            if not os.path.exists(batch_source):
                print(f"\n⚠️ Batch {batch_name} not found, skipping...")
                continue
            
            os.makedirs(batch_dest, exist_ok=True)
            print(f"\n📁 Processing {batch_name}/")
            
            # Iterate through years in this batch
            for year_dir in sorted(os.listdir(batch_source)):
                if year_dir.startswith("year_"):
                    year_source = f"{batch_source}/{year_dir}"
                    year_dest = f"{batch_dest}/{year_dir}"
                    os.makedirs(year_dest, exist_ok=True)
                    
                    print(f"\n   📂 {year_dir}/")
                    
                    # Process orders.parquet
                    orders_file = f"{year_source}/orders.parquet"
                    if os.path.exists(orders_file):
                        try:
                            df_orders = spark.read.parquet(orders_file)
                            original_orders = df_orders.count()
                            
                            # Apply corruption to orders
                            df_orders_corrupted = apply_selected_error_types(df_orders, 'orders')
                            corrupted_orders = df_orders_corrupted.count()
                            
                            # Save corrupted orders as single parquet file
                            orders_output = f"{year_dest}/orders.parquet"
                            write_single_parquet(df_orders_corrupted, orders_output)
                            
                            print(f"      Orders: {original_orders:,} → {corrupted_orders:,} (+{corrupted_orders - original_orders:,})")
                            
                            corruption_stats.append({
                                "layer": f"increments/{batch_name}/{year_dir}",
                                "file": "orders.parquet",
                                "original": original_orders,
                                "corrupted": corrupted_orders,
                                "increase": corrupted_orders - original_orders
                            })
                            
                        except Exception as e:
                            print(f"      ❌ Error processing orders in {year_dir}: {e}")
                    
                    # Process lineitem.parquet
                    lineitem_file = f"{year_source}/lineitem.parquet"
                    if os.path.exists(lineitem_file):
                        try:
                            df_lineitem = spark.read.parquet(lineitem_file)
                            original_lineitems = df_lineitem.count()
                            
                            # Apply corruption to lineitem
                            df_lineitem_corrupted = apply_selected_error_types(df_lineitem, 'lineitem')
                            corrupted_lineitems = df_lineitem_corrupted.count()
                            
                            # Save corrupted lineitem as single parquet file
                            lineitem_output = f"{year_dest}/lineitem.parquet"
                            write_single_parquet(df_lineitem_corrupted, lineitem_output)
                            
                            print(f"      Lineitems: {original_lineitems:,} → {corrupted_lineitems:,} (+{corrupted_lineitems - original_lineitems:,})")
                            
                            corruption_stats.append({
                                "layer": f"increments/{batch_name}/{year_dir}",
                                "file": "lineitem.parquet",
                                "original": original_lineitems,
                                "corrupted": corrupted_lineitems,
                                "increase": corrupted_lineitems - original_lineitems
                            })
                            
                        except Exception as e:
                            print(f"      ❌ Error processing lineitem in {year_dir}: {e}")
    
    # Print summary statistics (fixed variable name to avoid conflict with sum function)
    print("\n" + "="*80)
    print("📊 CORRUPTION SUMMARY")
    print("="*80)
    
    total_original = 0
    total_corrupted = 0
    
    for stat in corruption_stats:
        total_original += stat['original']
        total_corrupted += stat['corrupted']
    
    total_increase = total_corrupted - total_original
    
    print(f"\n{'Layer':<35} {'File':<20} {'Original':<12} {'Corrupted':<12} {'Increase':<12}")
    print("-" * 95)
    
    for stat in corruption_stats:
        print(f"{stat['layer']:<35} {stat['file']:<20} {stat['original']:<12,} {stat['corrupted']:<12,} +{stat['increase']:<11,}")
    
    print("-" * 95)
    print(f"{'TOTAL':<35} {'':<20} {total_original:<12,} {total_corrupted:<12,} +{total_increase:<11,}")
    
    print("\n" + "="*80)
    print("✅ DATA CORRUPTION COMPLETE!")
    print(f"   📁 Corrupted data saved to: {corrupted_path}")
    print("="*80)
    
    # Optional: Verify the corrupted data
    print("\n🔍 VERIFICATION - Reading corrupted files")
    print("="*80)
    
    # Verify base layer
    base_orders_path = f"{corrupted_bases_path}/orders_base_60.parquet"
    base_lineitem_path = f"{corrupted_bases_path}/lineitem_base_60.parquet"
    
    if os.path.exists(base_orders_path):
        try:
            df_check = spark.read.parquet(base_orders_path)
            print(f"✅ Successfully read corrupted orders_base_60.parquet: {df_check.count():,} records")
            
            # Check for introduced nulls
            null_count = df_check.filter(col("o_comment").isNull()).count()
            print(f"   NULLs in o_comment: {null_count:,} ({null_count/df_check.count()*100:.2f}%)")
            
            # Check for special characters
            special_count = df_check.filter(col("o_comment").rlike(".*[@#$%&*!?].*")).count()
            print(f"   Special characters in o_comment: {special_count:,} ({special_count/df_check.count()*100:.2f}%)")
            
        except Exception as e:
            print(f"❌ Error reading corrupted base orders: {e}")
    
    return corruption_stats

# Execute the corruption
if __name__ == "__main__":
    corruption_stats = corrupt_incremental_data()
    print("\n✨ Data corruption pipeline completed successfully!")


🔧 CORRUPTING INCREMENTAL INGESTION COMBINED DATA
Source: ../../../data/raw/tables/incremental_ingestion_combined
Destination: ../../../data/raw/tables/corrupted_incremental

📊 PROCESSING BASE LAYER

📁 Processing base file: lineitem_base_60.parquet
   Table type: lineitem
   Original records: 3,600,847
   ✅ Corrupted data saved to: ../../../data/raw/tables/corrupted_incremental/bases/lineitem_base_60.parquet
   Records: 3,600,847 → 3,708,231 (+107,384)

📁 Processing base file: orders_base_60.parquet
   Table type: orders
   Original records: 900,006
   ✅ Corrupted data saved to: ../../../data/raw/tables/corrupted_incremental/bases/orders_base_60.parquet
   Records: 900,006 → 927,067 (+27,061)

📊 PROCESSING INCREMENTAL BATCHES

📁 Processing batch_1/

   📂 year_1995/
      Orders: 9,962 → 10,290 (+328)
      Lineitems: 39,831 → 41,043 (+1,212)

   📂 year_1996/
      Orders: 189,787 → 195,533 (+5,746)
      Lineitems: 759,771 → 782,571 (+22,800)

📁 Processing batch_2/

   📂 year_1996/
   

In [9]:
def verify_corrupted_incremental_integrity():
    """Verify the integrity of corrupted_incremental data with the new file structure"""
    
    import os
    
    # Path to corrupted incremental data
    corrupted_path = "../../../data/raw/tables/corrupted_incremental"
    
    if not os.path.exists(corrupted_path):
        print(f"❌ Error: Corrupted path {corrupted_path} does not exist!")
        return
    
    print("\n" + "="*80)
    print("🔍 DATA INTEGRITY VERIFICATION - CORRUPTED INCREMENTAL")
    print("="*80)
    print(f"Path: {corrupted_path}")
    
    verification_results = []
    total_records_all = 0
    
    # 1. Verify Base Layer
    print("\n" + "="*60)
    print("📊 BASE LAYER VERIFICATION")
    print("="*60)
    
    bases_path = f"{corrupted_path}/bases"
    
    if os.path.exists(bases_path):
        for file_name in os.listdir(bases_path):
            if file_name.endswith('.parquet'):
                file_path = f"{bases_path}/{file_name}"
                
                print(f"\n📁 Base File: {file_name}")
                
                try:
                    # Read the corrupted file
                    df = spark.read.parquet(file_path)
                    
                    # Basic statistics
                    total_records = df.count()
                    distinct_records = df.distinct().count()
                    duplicate_records = total_records - distinct_records
                    
                    # Get column information
                    columns = df.columns
                    null_counts = {}
                    for col_name in columns:
                        null_count = df.filter(df[col_name].isNull()).count()
                        if null_count > 0:
                            null_counts[col_name] = null_count
                    
                    # Get sample of records with issues
                    issue_samples = {}
                    
                    # Check for special characters based on table type
                    if 'orders' in file_name and 'o_comment' in columns:
                        special_chars_df = df.filter(df["o_comment"].rlike(".*[@#$%&*!?].*"))
                        special_count = special_chars_df.count()
                        if special_count > 0:
                            issue_samples['special_chars'] = special_chars_df.limit(3).collect()
                        
                        # Check for leading/trailing spaces
                        spaces_df = df.filter((df["o_comment"].like(" %")) | (df["o_comment"].like("% ")))
                        spaces_count = spaces_df.count()
                        if spaces_count > 0:
                            issue_samples['spaces'] = spaces_df.limit(3).collect()
                    
                    if 'lineitem' in file_name and 'l_comment' in columns:
                        special_chars_df = df.filter(df["l_comment"].rlike(".*[@#$%&*!?].*"))
                        special_count = special_chars_df.count()
                        if special_count > 0:
                            issue_samples['special_chars'] = special_chars_df.limit(3).collect()
                        
                        # Check for leading/trailing spaces
                        spaces_df = df.filter((df["l_comment"].like(" %")) | (df["l_comment"].like("% ")))
                        spaces_count = spaces_df.count()
                        if spaces_count > 0:
                            issue_samples['spaces'] = spaces_df.limit(3).collect()
                    
                    # Print results
                    print(f"  ✅ Total records: {total_records:,}")
                    print(f"  ✅ Distinct records: {distinct_records:,}")
                    if duplicate_records > 0:
                        print(f"  ⚠️ Duplicate records: {duplicate_records:,} ({duplicate_records/total_records*100:.2f}%)")
                    
                    if null_counts:
                        print(f"  ⚠️ NULL values found:")
                        for col_name, count in null_counts.items():
                            print(f"      - {col_name}: {count:,} ({count/total_records*100:.2f}%)")
                    
                    # Check for specific corruption types in orders table
                    if 'orders' in file_name and 'o_totalprice' in columns:
                        # Check o_totalprice for negative or extreme values
                        negative_price = df.filter(df["o_totalprice"] < 0).count()
                        if negative_price > 0:
                            print(f"  ⚠️ Negative o_totalprice: {negative_price:,} records")
                        
                        # Check for case inconsistency in o_clerk
                        if 'o_clerk' in columns:
                            clerk_upper = df.filter(df["o_clerk"].rlike("^[A-Z]")).count()
                            clerk_lower = df.filter(df["o_clerk"].rlike("^[a-z]")).count()
                            if clerk_upper > 0 and clerk_lower > 0:
                                print(f"  ⚠️ Case inconsistency in o_clerk: {clerk_upper:,} uppercase, {clerk_lower:,} lowercase")
                    
                    # Check for specific corruption types in lineitem table
                    if 'lineitem' in file_name:
                        if 'l_quantity' in columns:
                            negative_quantity = df.filter(df["l_quantity"] < 0).count()
                            if negative_quantity > 0:
                                print(f"  ⚠️ Negative l_quantity: {negative_quantity:,} records")
                        
                        if 'l_discount' in columns:
                            invalid_discount = df.filter((df["l_discount"] < 0) | (df["l_discount"] > 0.1)).count()
                            if invalid_discount > 0:
                                print(f"  ⚠️ Invalid l_discount (outside 0-0.1): {invalid_discount:,} records")
                        
                        if 'l_tax' in columns:
                            invalid_tax = df.filter((df["l_tax"] < 0) | (df["l_tax"] > 0.08)).count()
                            if invalid_tax > 0:
                                print(f"  ⚠️ Invalid l_tax (outside 0-0.08): {invalid_tax:,} records")
                    
                    # Verify file is readable and has correct schema
                    sample = df.limit(3).toPandas()
                    print(f"  ✅ Sample records: {len(sample)} rows readable")
                    
                    # Show sample of corrupted data
                    if issue_samples:
                        print(f"  📝 Sample corrupted records:")
                        for issue_type, samples in issue_samples.items():
                            for row in samples[:2]:  # Show up to 2 samples
                                if 'o_comment' in row:
                                    comment_preview = str(row['o_comment'])[:80] + "..." if len(str(row['o_comment'])) > 80 else str(row['o_comment'])
                                    print(f"      - {issue_type}: {comment_preview}")
                                elif 'l_comment' in row:
                                    comment_preview = str(row['l_comment'])[:80] + "..." if len(str(row['l_comment'])) > 80 else str(row['l_comment'])
                                    print(f"      - {issue_type}: {comment_preview}")
                    
                    verification_results.append({
                        "layer": "base",
                        "file": file_name,
                        "total_records": total_records,
                        "duplicates": duplicate_records,
                        "nulls": len(null_counts),
                        "status": "OK"
                    })
                    
                    total_records_all += total_records
                    
                except Exception as e:
                    print(f"  ❌ Error reading {file_name}: {e}")
                    verification_results.append({
                        "layer": "base",
                        "file": file_name,
                        "status": "ERROR",
                        "error": str(e)
                    })
    else:
        print(f"⚠️ Base path not found: {bases_path}")
    
    # 2. Verify Incremental Batches
    print("\n" + "="*60)
    print("📊 INCREMENTAL BATCHES VERIFICATION")
    print("="*60)
    
    increments_path = f"{corrupted_path}/increments"
    
    if os.path.exists(increments_path):
        batch_count = 0
        year_count = 0
        
        # Iterate through batches
        for batch_num in range(1, 4):
            batch_name = f"batch_{batch_num}"
            batch_path = f"{increments_path}/{batch_name}"
            
            if not os.path.exists(batch_path):
                print(f"\n⚠️ Batch {batch_name} not found")
                continue
            
            batch_count += 1
            print(f"\n📁 Batch {batch_num}: {batch_name}/")
            
            # Iterate through years in this batch
            for year_dir in sorted(os.listdir(batch_path)):
                if year_dir.startswith("year_"):
                    year_path = f"{batch_path}/{year_dir}"
                    year_count += 1
                    
                    print(f"\n   📂 {year_dir}/")
                    
                    # Process orders.parquet
                    orders_file = f"{year_path}/orders.parquet"
                    if os.path.exists(orders_file):
                        try:
                            df_orders = spark.read.parquet(orders_file)
                            orders_count = df_orders.count()
                            distinct_orders = df_orders.distinct().count()
                            orders_duplicates = orders_count - distinct_orders
                            
                            # Check for nulls in comments
                            orders_nulls = df_orders.filter(df_orders["o_comment"].isNull()).count() if 'o_comment' in df_orders.columns else 0
                            
                            print(f"      Orders:")
                            print(f"        ✅ Total: {orders_count:,}")
                            if orders_duplicates > 0:
                                print(f"        ⚠️ Duplicates: {orders_duplicates:,}")
                            if orders_nulls > 0:
                                print(f"        ⚠️ NULL comments: {orders_nulls:,} ({orders_nulls/orders_count*100:.2f}%)")
                            
                            # Check for special characters in comments
                            special_orders = df_orders.filter(df_orders["o_comment"].rlike(".*[@#$%&*!?].*")).count() if 'o_comment' in df_orders.columns else 0
                            if special_orders > 0:
                                print(f"        ⚠️ Special chars in comments: {special_orders:,} ({special_orders/orders_count*100:.2f}%)")
                            
                            total_records_all += orders_count
                            
                            verification_results.append({
                                "layer": f"increments/{batch_name}/{year_dir}",
                                "file": "orders.parquet",
                                "total_records": orders_count,
                                "duplicates": orders_duplicates,
                                "nulls": orders_nulls,
                                "special_chars": special_orders,
                                "status": "OK"
                            })
                            
                        except Exception as e:
                            print(f"        ❌ Error reading orders: {e}")
                            verification_results.append({
                                "layer": f"increments/{batch_name}/{year_dir}",
                                "file": "orders.parquet",
                                "status": "ERROR",
                                "error": str(e)
                            })
                    
                    # Process lineitem.parquet
                    lineitem_file = f"{year_path}/lineitem.parquet"
                    if os.path.exists(lineitem_file):
                        try:
                            df_lineitem = spark.read.parquet(lineitem_file)
                            lineitem_count = df_lineitem.count()
                            distinct_lineitems = df_lineitem.distinct().count()
                            lineitem_duplicates = lineitem_count - distinct_lineitems
                            
                            # Check for nulls in comments
                            lineitem_nulls = df_lineitem.filter(df_lineitem["l_comment"].isNull()).count() if 'l_comment' in df_lineitem.columns else 0
                            
                            print(f"      Lineitems:")
                            print(f"        ✅ Total: {lineitem_count:,}")
                            if lineitem_duplicates > 0:
                                print(f"        ⚠️ Duplicates: {lineitem_duplicates:,}")
                            if lineitem_nulls > 0:
                                print(f"        ⚠️ NULL comments: {lineitem_nulls:,} ({lineitem_nulls/lineitem_count*100:.2f}%)")
                            
                            # Check for special characters in comments
                            special_lineitems = df_lineitem.filter(df_lineitem["l_comment"].rlike(".*[@#$%&*!?].*")).count() if 'l_comment' in df_lineitem.columns else 0
                            if special_lineitems > 0:
                                print(f"        ⚠️ Special chars in comments: {special_lineitems:,} ({special_lineitems/lineitem_count*100:.2f}%)")
                            
                            # Check for numeric corruptions
                            if 'l_quantity' in df_lineitem.columns:
                                negative_quantity = df_lineitem.filter(df_lineitem["l_quantity"] < 0).count()
                                if negative_quantity > 0:
                                    print(f"        ⚠️ Negative quantity: {negative_quantity:,} records")
                            
                            total_records_all += lineitem_count
                            
                            verification_results.append({
                                "layer": f"increments/{batch_name}/{year_dir}",
                                "file": "lineitem.parquet",
                                "total_records": lineitem_count,
                                "duplicates": lineitem_duplicates,
                                "nulls": lineitem_nulls,
                                "special_chars": special_lineitems,
                                "status": "OK"
                            })
                            
                        except Exception as e:
                            print(f"        ❌ Error reading lineitem: {e}")
                            verification_results.append({
                                "layer": f"increments/{batch_name}/{year_dir}",
                                "file": "lineitem.parquet",
                                "status": "ERROR",
                                "error": str(e)
                            })
        
        print(f"\n📊 Summary:")
        print(f"   Batches processed: {batch_count}")
        print(f"   Years processed: {year_count}")
        
    else:
        print(f"⚠️ Increments path not found: {increments_path}")
    
    # 3. Overall Summary
    print("\n" + "="*80)
    print("📊 OVERALL INTEGRITY SUMMARY")
    print("="*80)
    
    successful_files = [r for r in verification_results if r.get("status") == "OK"]
    failed_files = [r for r in verification_results if r.get("status") == "ERROR"]
    
    # FIXED: Use manual sum instead of sum() function to avoid conflict
    total_duplicates = 0
    total_nulls = 0
    total_special_chars = 0
    
    for r in successful_files:
        total_duplicates += r.get("duplicates", 0)
        total_nulls += r.get("nulls", 0)
        total_special_chars += r.get("special_chars", 0)
    
    print(f"\n✅ Successful files: {len(successful_files)}")
    if failed_files:
        print(f"❌ Failed files: {len(failed_files)}")
        for failed in failed_files:
            print(f"   - {failed['layer']}/{failed['file']}: {failed.get('error', 'Unknown error')}")
    
    if total_records_all > 0:
        print(f"\n📈 Corruption Statistics:")
        print(f"   Total records processed: {total_records_all:,}")
        print(f"   Total duplicates found: {total_duplicates:,} ({total_duplicates/total_records_all*100:.2f}% of total)")
        print(f"   Total NULL values found: {total_nulls:,} ({total_nulls/total_records_all*100:.2f}% of total)")
        print(f"   Total special characters found: {total_special_chars:,} ({total_special_chars/total_records_all*100:.2f}% of total)")
    
    # 4. Data Quality Assessment
    print("\n" + "="*80)
    print("📊 DATA QUALITY ASSESSMENT")
    print("="*80)
    
    quality_score = 100
    
    if total_records_all > 0:
        # Deduct for duplicates (max 20 points)
        duplicate_rate = total_duplicates / total_records_all
        if duplicate_rate > 0.05:
            quality_score -= 20
        elif duplicate_rate > 0.03:
            quality_score -= 15
        elif duplicate_rate > 0.01:
            quality_score -= 10
        elif duplicate_rate > 0:
            quality_score -= 5
        
        # Deduct for nulls (max 15 points)
        null_rate = total_nulls / total_records_all
        if null_rate > 0.05:
            quality_score -= 15
        elif null_rate > 0.03:
            quality_score -= 10
        elif null_rate > 0.01:
            quality_score -= 5
        elif null_rate > 0:
            quality_score -= 2
        
        # Deduct for special characters (max 10 points)
        special_rate = total_special_chars / total_records_all
        if special_rate > 0.15:
            quality_score -= 10
        elif special_rate > 0.10:
            quality_score -= 7
        elif special_rate > 0.05:
            quality_score -= 5
        elif special_rate > 0:
            quality_score -= 2
    
    print(f"\n📊 Overall Data Quality Score: {quality_score:.1f}/100")
    
    if quality_score >= 80:
        print("   ✅ Good: Data corruption successfully applied with expected quality issues")
    elif quality_score >= 60:
        print("   ⚠️ Fair: Some corruption applied, but may need more aggressive injection")
    else:
        print("   ❌ Poor: High level of corruption detected as expected")
    
    print("\n" + "="*80)
    print("✅ INTEGRITY VERIFICATION COMPLETE!")
    print("="*80)
    
    return verification_results, total_records_all

# Run integrity check
verification_results, total_records = verify_corrupted_incremental_integrity()


🔍 DATA INTEGRITY VERIFICATION - CORRUPTED INCREMENTAL
Path: ../../../data/raw/tables/corrupted_incremental

📊 BASE LAYER VERIFICATION

📁 Base File: lineitem_base_60.parquet
  ✅ Total records: 3,708,231
  ✅ Distinct records: 3,612,551
  ⚠️ Duplicate records: 95,680 (2.58%)
  ⚠️ NULL values found:
      - l_quantity: 16,821 (0.45%)
      - l_discount: 16,999 (0.46%)
      - l_tax: 16,989 (0.46%)
      - l_comment: 111,150 (3.00%)
  ⚠️ Negative l_quantity: 32,780 records
  ⚠️ Invalid l_discount (outside 0-0.1): 46,341 records
  ⚠️ Invalid l_tax (outside 0-0.08): 46,173 records
  ✅ Sample records: 3 rows readable
  📝 Sample corrupted records:
      - special_chars:  excuses haggle carefully. pe  @
      - special_chars: kly even pac&
      - spaces: ests. ironic, pending requests lose slyly 
      - spaces:  sleep furiously regular foxes.

📁 Base File: orders_base_60.parquet
  ✅ Total records: 927,067
  ✅ Distinct records: 901,075
  ⚠️ Duplicate records: 25,992 (2.80%)
  ⚠️ NULL values fo

In [10]:
import shutil
import os
import glob

def cleanup_combined_data():
    """Delete the incremental_ingestion_combined folder after processing"""
    
    # Path to combined folder
    combined_path = "../../../data/raw/tables/incremental_ingestion_combined"
    
    print("\n" + "="*80)
    print("🧹 CLEANUP: Removing Combined Incremental Ingestion Data")
    print("="*80)
    
    # Check if combined folder exists
    if not os.path.exists(combined_path):
        print(f"❌ ERROR: Combined folder not found at {combined_path}")
        print("   Nothing to delete!")
        return False
    
    # Show folder size before deletion
    print("\n📊 BEFORE CLEANUP:")
    
    # Calculate combined folder size
    combined_size = 0
    combined_file_count = 0
    combined_dir_count = 0
    
    for root, dirs, files in os.walk(combined_path):
        combined_dir_count += 1
        for file in files:
            file_path = os.path.join(root, file)
            combined_size += os.path.getsize(file_path)
            combined_file_count += 1
    
    combined_size_mb = combined_size / (1024 * 1024)
    combined_size_gb = combined_size_mb / 1024
    
    print(f"   Combined folder: {combined_path}")
    print(f"   Size: {combined_size_gb:.2f} GB ({combined_size_mb:.2f} MB)")
    print(f"   Files: {combined_file_count:,}")
    print(f"   Directories: {combined_dir_count:,}")
    
    # Show structure preview
    print("\n📁 DIRECTORY STRUCTURE PREVIEW:")
    
    def print_structure(path, max_depth=2, current_depth=0):
        """Print directory structure preview"""
        if current_depth > max_depth or not os.path.exists(path):
            return
        
        items = sorted(os.listdir(path))
        for i, item in enumerate(items[:10]):  # Show up to 10 items per level
            item_path = os.path.join(path, item)
            indent = "  " * current_depth
            
            if os.path.isdir(item_path):
                print(f"{indent}📁 {item}/")
                print_structure(item_path, max_depth, current_depth + 1)
            else:
                if item.endswith('.parquet'):
                    size_mb = os.path.getsize(item_path) / (1024 * 1024)
                    print(f"{indent}📄 {item} ({size_mb:.2f} MB)")
        
        if len(items) > 10:
            print(f"{'  ' * current_depth}... and {len(items) - 10} more items")
    
    print_structure(combined_path, max_depth=2)
    
    # Confirmation
    print("\n" + "="*80)
    print("⚠️  WARNING: This will permanently delete the incremental_ingestion_combined folder!")
    print(f"   Path: {combined_path}")
    print(f"   Size to delete: {combined_size_gb:.2f} GB")
    print("="*80)
    
    # Ask for confirmation
    auto_confirm = False  # Set to True only for automation
    
    if auto_confirm:
        print("\n🔄 Auto-confirming deletion (for automation)...")
        confirm = 'yes'
    else:
        print("\n⚠️  This action cannot be undone!")
        confirm = input("Type 'yes' to confirm deletion: ")
    
    if confirm.lower() == 'yes':
        try:
            # Delete the combined folder
            print(f"\n🗑️  Deleting: {combined_path}")
            shutil.rmtree(combined_path)
            print("   ✅ Combined folder deleted successfully!")
            
            # Clean up any temporary files that might be left
            temp_patterns = [
                "../../../data/raw/tables/temp_*",
                "../../../data/raw/tables/*/temp_*",
                "../../../data/raw/tables/*/*/temp_*"
            ]
            
            temp_files_removed = 0
            for temp_pattern in temp_patterns:
                for temp_file in glob.glob(temp_pattern):
                    try:
                        if os.path.isfile(temp_file):
                            os.remove(temp_file)
                            temp_files_removed += 1
                        elif os.path.isdir(temp_file):
                            shutil.rmtree(temp_file)
                            temp_files_removed += 1
                    except Exception as e:
                        print(f"   ⚠️  Could not remove {temp_file}: {e}")
            
            if temp_files_removed > 0:
                print(f"   🗑️  Removed {temp_files_removed} temporary files/directories")
            
            # Final verification
            print("\n" + "="*80)
            print("✅ CLEANUP COMPLETE!")
            print("="*80)
            
            # Verify deletion
            if not os.path.exists(combined_path):
                print(f"\n✅ Successfully deleted: {combined_path}")
                print(f"   Space freed: {combined_size_gb:.2f} GB")
                
                # Show what's left in the parent directory
                parent_path = "../../../data/raw/tables/"
                if os.path.exists(parent_path):
                    print(f"\n📁 Remaining contents in {parent_path}:")
                    remaining_items = sorted([item for item in os.listdir(parent_path) 
                                            if os.path.isdir(os.path.join(parent_path, item))])
                    for item in remaining_items:
                        item_path = os.path.join(parent_path, item)
                        # Calculate size for remaining folders
                        folder_size = 0
                        folder_files = 0
                        for root, dirs, files in os.walk(item_path):
                            for file in files:
                                file_path = os.path.join(root, file)
                                folder_size += os.path.getsize(file_path)
                                folder_files += 1
                        
                        folder_size_mb = folder_size / (1024 * 1024)
                        folder_size_gb = folder_size_mb / 1024
                        print(f"   📁 {item}/ - {folder_size_gb:.2f} GB, {folder_files} files")
            else:
                print(f"⚠️ Warning: {combined_path} still exists after deletion attempt!")
            
            return True
            
        except Exception as e:
            print(f"\n❌ ERROR during cleanup: {e}")
            print("   The folder may not have been fully deleted.")
            import traceback
            traceback.print_exc()
            return False
    else:
        print("\n❌ Deletion cancelled by user.")
        print(f"   Combined folder was NOT deleted: {combined_path}")
        return False

# Execute cleanup
print("\n" + "="*80)
print("🚀 STARTING COMBINED DATA CLEANUP")
print("="*80)

cleanup_successful = cleanup_combined_data()

if cleanup_successful:
    print("\n" + "="*80)
    print("✨ CLEANUP COMPLETED SUCCESSFULLY!")
    print("="*80)
    print("   ✅ Combined folder deleted")
    print("   ✅ Disk space reclaimed")
    print("="*80)
else:
    print("\n" + "="*80)
    print("⚠️ CLEANUP WAS NOT COMPLETED")
    print("="*80)
    print("   Please check the errors above and verify:")
    print("   1. The folder path is correct")
    print("   2. You have proper permissions to delete")
    print("   3. No processes are using the folder")
    print("="*80)


🚀 STARTING COMBINED DATA CLEANUP

🧹 CLEANUP: Removing Combined Incremental Ingestion Data

📊 BEFORE CLEANUP:
   Combined folder: ../../../data/raw/tables/incremental_ingestion_combined
   Size: 0.26 GB (261.55 MB)
   Files: 14
   Directories: 12

📁 DIRECTORY STRUCTURE PREVIEW:
📁 bases/
  📄 lineitem_base_60.parquet (125.70 MB)
  📄 orders_base_60.parquet (32.78 MB)
📁 increments/
  📁 batch_1/
    📁 year_1995/
    📁 year_1996/
  📁 batch_2/
    📁 year_1996/
    📁 year_1997/
  📁 batch_3/
    📁 year_1997/
    📁 year_1998/

⚠️  WARNING: This will permanently delete the incremental_ingestion_combined folder!
   Path: ../../../data/raw/tables/incremental_ingestion_combined
   Size to delete: 0.26 GB

⚠️  This action cannot be undone!


Type 'yes' to confirm deletion:  yes



🗑️  Deleting: ../../../data/raw/tables/incremental_ingestion_combined
   ✅ Combined folder deleted successfully!

✅ CLEANUP COMPLETE!

✅ Successfully deleted: ../../../data/raw/tables/incremental_ingestion_combined
   Space freed: 0.26 GB

📁 Remaining contents in ../../../data/raw/tables/:
   📁 .ipynb_checkpoints/ - 0.00 GB, 0 files
   📁 corrupted_incremental/ - 0.28 GB, 14 files
   📁 original/ - 0.37 GB, 8 files

✨ CLEANUP COMPLETED SUCCESSFULLY!
   ✅ Combined folder deleted
   ✅ Disk space reclaimed


In [11]:
import os
import shutil

def rename_corrupted_folder():
    """Rename corrupted_incremental to a more professional name"""
    
    current_path = "../../../data/raw/tables/corrupted_incremental"
    
    new_name = "work_data"  
    
    new_path = f"../../../data/raw/tables/{new_name}"
    
    print("\n" + "="*80)
    print("📁 RENAMING FINAL DATA FOLDER")
    print("="*80)
    
    if not os.path.exists(current_path):
        print(f"❌ Error: {current_path} does not exist!")
        return False
    
    if os.path.exists(new_path):
        print(f"⚠️ Warning: {new_path} already exists!")
        response = input("Do you want to overwrite? (yes/no): ")
        if response.lower() != 'yes':
            print("❌ Rename cancelled.")
            return False
        shutil.rmtree(new_path)
    
    # Rename the folder
    print(f"\n🔄 Renaming:")
    print(f"   From: {current_path}")
    print(f"   To:   {new_path}")
    
    os.rename(current_path, new_path)
    
    # Verify the rename
    if os.path.exists(new_path) and not os.path.exists(current_path):
        print("\n✅ Rename successful!")
        
        # Show folder contents
        print(f"\n📁 Contents of {new_name}:")
        for item in sorted(os.listdir(new_path))[:10]:
            item_path = os.path.join(new_path, item)
            if os.path.isdir(item_path):
                # Calculate size for directories
                dir_size = 0
                for root, dirs, files in os.walk(item_path):
                    for file in files:
                        dir_size += os.path.getsize(os.path.join(root, file))
                dir_size_mb = dir_size / (1024 * 1024)
                print(f"   📁 {item}/ - {dir_size_mb:.2f} MB")
            else:
                if item.endswith('.parquet'):
                    size_mb = os.path.getsize(item_path) / (1024 * 1024)
                    print(f"   📄 {item} ({size_mb:.2f} MB)")
        
        return True
    else:
        print("❌ Rename failed!")
        return False

# Execute rename
rename_corrupted_folder()


📁 RENAMING FINAL DATA FOLDER

🔄 Renaming:
   From: ../../../data/raw/tables/corrupted_incremental
   To:   ../../../data/raw/tables/work_data

✅ Rename successful!

📁 Contents of work_data:
   📁 bases/ - 172.95 MB
   📁 increments/ - 111.76 MB


True

In [12]:
def add_original_data_to_work_data():
    """Add all original tables (orders and lineitem) from original folder to work_data"""
    
    import shutil
    import os
    
    # Define paths
    work_data_path = "../../../data/raw/tables/work_data"
    original_path = "/home/jovyan/data/raw/tables/original"
    original_data_folder = "original_data"
    
    print("\n" + "="*80)
    print("📦 ADDING ORIGINAL DATA TO WORK_DATA")
    print("="*80)
    
    # Check if work_data exists
    if not os.path.exists(work_data_path):
        print(f"❌ Error: work_data folder not found at {work_data_path}")
        print("   Please run the rename operation first.")
        return False
    
    # Check if original path exists
    if not os.path.exists(original_path):
        print(f"❌ Error: Original data path not found at {original_path}")
        return False
    
    # Create original_data folder inside work_data
    original_data_folder_path = os.path.join(work_data_path, original_data_folder)
    if os.path.exists(original_data_folder_path):
        print(f"\n⚠️ Warning: {original_data_folder_path} already exists!")
        response = input("Do you want to overwrite it? (yes/no): ")
        if response.lower() != 'yes':
            print("❌ Operation cancelled.")
            return False
        shutil.rmtree(original_data_folder_path)
    
    os.makedirs(original_data_folder_path)
    print(f"\n✅ Created folder: {original_data_folder_path}")
    
    # Copy all files from original folder
    print("\n📁 Copying all files from original folder:")
    print("-" * 60)
    
    copied_files = []
    failed_files = []
    total_size = 0
    
    # Loop through all items in the original folder
    for item in os.listdir(original_path):
        source_path = os.path.join(original_path, item)
        dest_path = os.path.join(original_data_folder_path, item)
        
        try:
            if os.path.isfile(source_path):
                # Copy file
                file_size = os.path.getsize(source_path)
                file_size_mb = file_size / (1024 * 1024)
                total_size += file_size
                
                print(f"📄 Copying: {item} ({file_size_mb:.2f} MB)")
                shutil.copy2(source_path, dest_path)
                
                # Verify
                if os.path.exists(dest_path):
                    print(f"   ✅ Successfully copied")
                    copied_files.append({
                        "name": item,
                        "size": file_size,
                        "type": "file"
                    })
                else:
                    print(f"   ❌ Failed to copy")
                    failed_files.append(item)
                    
            elif os.path.isdir(source_path):
                # Copy entire directory
                print(f"📁 Copying directory: {item}/")
                
                # Get directory size before copying
                dir_size = 0
                for root, dirs, files in os.walk(source_path):
                    for file in files:
                        file_path = os.path.join(root, file)
                        dir_size += os.path.getsize(file_path)
                
                dir_size_mb = dir_size / (1024 * 1024)
                total_size += dir_size
                print(f"   Size: {dir_size_mb:.2f} MB")
                
                # Copy entire directory
                shutil.copytree(source_path, dest_path)
                
                # Verify
                if os.path.exists(dest_path):
                    print(f"   ✅ Successfully copied directory")
                    copied_files.append({
                        "name": item,
                        "size": dir_size,
                        "type": "directory"
                    })
                else:
                    print(f"   ❌ Failed to copy directory")
                    failed_files.append(item)
                    
        except Exception as e:
            print(f"   ❌ Error: {e}")
            failed_files.append(item)
    
    # Show summary
    print("\n" + "="*80)
    print("📊 COPY SUMMARY")
    print("="*80)
    
    if copied_files:
        total_size_mb = total_size / (1024 * 1024)
        total_size_gb = total_size_mb / 1024
        
        print(f"\n✅ Successfully copied: {len(copied_files)} items")
        print(f"   Total size: {total_size_gb:.2f} GB ({total_size_mb:.2f} MB)")
        print(f"\n📁 Items copied:")
        
        # Separate files and directories
        files_copied = [f for f in copied_files if f['type'] == 'file']
        dirs_copied = [f for f in copied_files if f['type'] == 'directory']
        
        if files_copied:
            print(f"\n   Files ({len(files_copied)}):")
            for f in files_copied:
                print(f"      - {f['name']} ({f['size']/(1024*1024):.2f} MB)")
        
        if dirs_copied:
            print(f"\n   Directories ({len(dirs_copied)}):")
            for d in dirs_copied:
                print(f"      - {d['name']}/ ({d['size']/(1024*1024):.2f} MB)")
    
    if failed_files:
        print(f"\n❌ Failed to copy: {len(failed_files)} items")
        for item in failed_files:
            print(f"   - {item}")
    
    # If no files were copied, return False
    if not copied_files:
        print("\n❌ No files or directories were copied.")
        return False
    
    # Read and verify the copied data using Spark
    print("\n" + "="*80)
    print("🔍 VERIFYING COPIED DATA WITH SPARK")
    print("="*80)
    
    verification_results = []
    
    # Find all parquet files in the copied directory
    all_parquet_files = []
    for root, dirs, files in os.walk(original_data_folder_path):
        for file in files:
            if file.endswith('.parquet'):
                all_parquet_files.append(os.path.join(root, file))
    
    print(f"\n📊 Found {len(all_parquet_files)} parquet files to verify")
    
    # Verify up to 10 files
    for i, file_path in enumerate(all_parquet_files[:10]):
        file_name = os.path.basename(file_path)
        rel_path = os.path.relpath(file_path, original_data_folder_path)
        
        print(f"\n📊 Verifying: {rel_path}")
        
        try:
            # Read the parquet file
            df = spark.read.parquet(file_path)
            
            # Get basic statistics
            record_count = df.count()
            column_count = len(df.columns)
            
            print(f"   ✅ File successfully read")
            print(f"   📊 Records: {record_count:,}")
            print(f"   📋 Columns: {column_count}")
            
            verification_results.append({
                "file": rel_path,
                "status": "OK",
                "records": record_count,
                "columns": column_count
            })
            
        except Exception as e:
            print(f"   ❌ Error reading {file_name}: {e}")
            verification_results.append({
                "file": rel_path,
                "status": "ERROR",
                "error": str(e)
            })
    
    # Show final work_data structure
    print("\n" + "="*80)
    print("📁 FINAL WORK_DATA STRUCTURE")
    print("="*80)
    
    def print_work_data_structure(path, max_depth=2, current_depth=0):
        """Print the complete structure of work_data"""
        if current_depth > max_depth or not os.path.exists(path):
            return
        
        indent = "  " * current_depth
        items = sorted(os.listdir(path))
        
        for item in items:
            item_path = os.path.join(path, item)
            
            if os.path.isdir(item_path):
                # Calculate folder size
                folder_size = 0
                folder_files = 0
                for root, dirs, files in os.walk(item_path):
                    for file in files:
                        file_path = os.path.join(root, file)
                        folder_size += os.path.getsize(file_path)
                        folder_files += 1
                
                folder_size_mb = folder_size / (1024 * 1024)
                folder_size_gb = folder_size_mb / 1024
                
                if folder_size_gb >= 1:
                    size_str = f"{folder_size_gb:.2f} GB"
                else:
                    size_str = f"{folder_size_mb:.2f} MB"
                
                print(f"{indent}📁 {item}/ - {size_str}, {folder_files} files")
                if current_depth < max_depth:
                    print_work_data_structure(item_path, max_depth, current_depth + 1)
            else:
                if item.endswith('.parquet'):
                    file_size = os.path.getsize(item_path)
                    file_size_mb = file_size / (1024 * 1024)
                    print(f"{indent}📄 {item} ({file_size_mb:.2f} MB)")
    
    print(f"\n📁 work_data/")
    print_work_data_structure(work_data_path, max_depth=2)
    
    return len(copied_files) > 0

# Execute after rename
print("\n" + "="*80)
print("🚀 ADDING ORIGINAL DATA TO WORK_DATA")
print("="*80)

original_data_added = add_original_data_to_work_data()

if original_data_added:
    print("\n✨ All operations completed successfully!")
    print("   ✓ Combined folder deleted")
    print("   ✓ Corrupted folder renamed to 'work_data'")
    print("   ✓ Original data copied to work_data/original_data/")
    print("="*80)
else:
    print("\n⚠️ Some operations had issues. Please check the logs above.")
    print("="*80)


🚀 ADDING ORIGINAL DATA TO WORK_DATA

📦 ADDING ORIGINAL DATA TO WORK_DATA

✅ Created folder: ../../../data/raw/tables/work_data/original_data

📁 Copying all files from original folder:
------------------------------------------------------------
📄 Copying: customer.parquet (12.06 MB)
   ✅ Successfully copied
📄 Copying: lineitem.parquet (262.03 MB)
   ✅ Successfully copied
📄 Copying: nation.parquet (0.00 MB)
   ✅ Successfully copied
📄 Copying: orders.parquet (58.96 MB)
   ✅ Successfully copied
📄 Copying: part.parquet (6.62 MB)
   ✅ Successfully copied
📄 Copying: partsupp.parquet (43.12 MB)
   ✅ Successfully copied
📄 Copying: region.parquet (0.00 MB)
   ✅ Successfully copied
📄 Copying: supplier.parquet (0.77 MB)
   ✅ Successfully copied

📊 COPY SUMMARY

✅ Successfully copied: 8 items
   Total size: 0.37 GB (383.56 MB)

📁 Items copied:

   Files (8):
      - customer.parquet (12.06 MB)
      - lineitem.parquet (262.03 MB)
      - nation.parquet (0.00 MB)
      - orders.parquet (58.96 MB)


In [13]:
# Remove lineitem and orders files from original_data
[os.remove(os.path.join(root, file)) for root, dirs, files in os.walk("../../../data/raw/tables/work_data/original_data") for file in files if file.endswith('.parquet') and ('lineitem' in file.lower() or 'orders' in file.lower())]
print(f"✅ Removed all lineitem and orders parquet files from original_data")

✅ Removed all lineitem and orders parquet files from original_data
